In [1]:
!git clone -b main https://github.com/vision4robotics/HiFT.git /kaggle/working/proj
%cd /kaggle/working/proj
!pip -q install yacs tqdm colorama pyyaml matplotlib opencv-python gdown openxlab

# Check the exact mount path of your attached UAV123 dataset
!ls -la /kaggle/input/

Cloning into '/kaggle/working/proj'...
remote: Enumerating objects: 376, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 376 (delta 0), reused 1 (delta 0), pack-reused 373 (from 1)
Receiving objects: 100% (376/376), 45.39 MiB | 27.78 MiB/s, done.
Resolving deltas: 100% (141/141), done.
/kaggle/working/proj
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.85.1 requires setuptools>=77.0.1, but you have setuptools 60.2.0 which is

In [4]:
!find /kaggle/input/datasets -maxdepth 4 -type d

/kaggle/input/datasets
/kaggle/input/datasets/galaxythereal
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV123
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/anno/UAV20L
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/data_seq
/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/data_seq/UAV123


In [5]:
!mkdir -p /tmp/test_dataset
!ln -sfn /kaggle/input/datasets/galaxythereal/uav123-tracking-dataset /tmp/test_dataset/UAV123
!ln -sfn /kaggle/input/datasets/galaxythereal/uav123-tracking-dataset /tmp/test_dataset/UAV20L
!ln -sfn /tmp/test_dataset /kaggle/working/proj/test_dataset

In [12]:
!rm -rf "/tmp/test_dataset/Crowd Dataset" /tmp/test_dataset/__MACOSX

In [18]:
!rm -f /tmp/UAV123_10fps.zip
!gdown "1soO5i5MNehu1PEfz1ng2AlK1ygd1qwp_" -O /tmp/UAV123_10fps.zip
!unzip -q /tmp/UAV123_10fps.zip -d /tmp/test_dataset/
!rm /tmp/UAV123_10fps.zip
!ls -la /tmp/test_dataset/

Downloading...
From: https://drive.google.com/uc?id=1soO5i5MNehu1PEfz1ng2AlK1ygd1qwp_
To: /tmp/UAV123_10fps.zip
100%|█████████████████████████████████████████| 608k/608k [00:00<00:00, 116MB/s]
[/tmp/UAV123_10fps.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /tmp/UAV123_10fps.zip or
        /tmp/UAV123_10fps.zip.zip, and cannot find /tmp/UAV123_10fps.zip.ZIP, period.
total 16
drwxr-xr-x 2 root root 4096 Nov 18 18:49 .
drwxrwxrwt 1 root root 4096 Nov 18 18:58 ..
lrwxrwxrwx 1 root root   60 Nov 18 18:39 UAV123 -> /kaggle/input/datasets/galaxythereal/uav123-tracking-dataset
lrwxrwxrwx 1 root root   60 Nov 18 18:39 UAV20L -> /kaggle/input/datasets/galaxythereal/uav123-tracking-dataset


In [20]:
!ls toolkit/datasets/
!grep -n "json\|data_seq\|anno" toolkit/datasets/uav.py | head -n 25

dataset.py  __init__.py  uav10fps.py  uav.py
dtb.py	    __pycache__  uav20l.py    video.py
2:import json
36:        with open(os.path.join(dataset_root, name+'.json'), 'r') as f:
37:            meta_data = json.load(f)


In [23]:
import os, re, json, shutil

SRC = '/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset'
DST = '/tmp/test_dataset'

# Clean old top-level symlinks so we can create real folders with JSONs/sub-symlinks
for name in ['UAV123', 'UAV20L', 'UAV123_20L', 'UAV123_10fps']:
    p = os.path.join(DST, name)
    if os.path.islink(p): os.unlink(p)
    elif os.path.isdir(p): shutil.rmtree(p)

# --- 1. Parse configSeqs.m for UAV123 & UAV123_10fps ---
cfg_text = open(f'{SRC}/configSeqs.m').read()
pattern = re.compile(
    r"struct\('name','([^']+)','path',\[[^\]]*'([^'/\\]+)[/\\]?'\],"
    r"'startFrame',(\d+),'endFrame',(\d+),'nz',(\d+),'ext','([^']+)'"
)
seqs = pattern.findall(cfg_text)
print(f"Parsed {len(seqs)} sequences from configSeqs.m")

# --- 2. Setup UAV123 + generate UAV123.json ---
uav123_dir = f'{DST}/UAV123'
os.makedirs(uav123_dir, exist_ok=True)
os.symlink(f'{SRC}/data_seq', f'{uav123_dir}/data_seq')
os.symlink(f'{SRC}/anno', f'{uav123_dir}/anno')

uav123_meta = {}
# --- 3. Setup UAV123_10fps (every 3rd frame + every 3rd annotation line) ---
uav10_dir = f'{DST}/UAV123_10fps'
os.makedirs(f'{uav10_dir}/data_seq', exist_ok=True)
os.makedirs(f'{uav10_dir}/anno', exist_ok=True)

for name, folder, start_f, end_f, nz, ext in seqs:
    start_f, end_f, nz = int(start_f), int(end_f), int(nz)
    anno_path = f'{SRC}/anno/UAV123/{name}.txt'
    if not os.path.exists(anno_path):
        continue
    raw_lines = [l.strip() for l in open(anno_path) if l.strip()]
    bboxes = []
    for l in raw_lines:
        vals = [float(x) for x in l.split(',')]
        bboxes.append([int(v) if v.is_integer() else v for v in vals])

    img_names = [f'data_seq/UAV123/{folder}/{i:0{nz}d}.{ext}' for i in range(start_f, end_f + 1)]
    n = min(len(bboxes), len(img_names))
    bboxes, img_names, raw_lines = bboxes[:n], img_names[:n], raw_lines[:n]

    uav123_meta[name] = {
        'video_dir': folder,
        'init_rect': bboxes[0],
        'img_names': img_names,
        'gt_rect': bboxes,
        'attr': []
    }

    # Subsample 3x for UAV123_10fps
    seq10_dir = f'{uav10_dir}/data_seq/{name}'
    os.makedirs(seq10_dir, exist_ok=True)
    sub_lines = raw_lines[::3]
    sub_imgs = img_names[::3]
    with open(f'{uav10_dir}/anno/{name}.txt', 'w') as f:
        f.write('\n'.join(sub_lines) + '\n')
    for idx, rel_img in enumerate(sub_imgs, 1):
        os.symlink(f'{SRC}/{rel_img}', f'{seq10_dir}/{idx:0{nz}d}.{ext}')

with open(f'{uav123_dir}/UAV123.json', 'w') as f:
    json.dump(uav123_meta, f)

# --- 4. Setup UAV123_20L (only the 20 sequences in anno/UAV20L) ---
uav20_dir = f'{DST}/UAV123_20L'
os.makedirs(f'{uav20_dir}/data_seq', exist_ok=True)
os.symlink(f'{SRC}/anno/UAV20L', f'{uav20_dir}/anno')

for txt in sorted(os.listdir(f'{SRC}/anno/UAV20L')):
    if txt.endswith('.txt'):
        seq_name = txt[:-4]
        src_seq = f'{SRC}/data_seq/UAV123/{seq_name}'
        if os.path.isdir(src_seq):
            os.symlink(src_seq, f'{uav20_dir}/data_seq/{seq_name}')

# --- 5. Verify via HiFT's DatasetFactory ---
from toolkit.datasets import DatasetFactory
for ds_name, ds_root in [
    ('UAV123', './test_dataset/UAV123'),
    ('UAV20l', './test_dataset/UAV123_20L/data_seq'),
    ('UAV10fps', './test_dataset/UAV123_10fps/data_seq'),
]:
    ds = DatasetFactory.create_dataset(name=ds_name, dataset_root=ds_root, load_img=False)
    print(f"✅ {ds_name}: loaded {len(ds)} videos")

Parsed 0 sequences from configSeqs.m


loading UAV123: 0it [00:00, ?it/s]


✅ UAV123: loaded 0 videos


loading UAV20l: 100%|█████████████████████████████████████████| 20/20 [00:01<00:00, 11.99it/s, uav1]


✅ UAV20l: loaded 20 videos


loading UAV10fps: 0it [00:00, ?it/s]

✅ UAV10fps: loaded 0 videos


In [24]:
!head -n 20 /kaggle/input/datasets/galaxythereal/uav123-tracking-dataset/configSeqs.m
!grep -n "path=" toolkit/datasets/uav10fps.py

function seqs=configSeqs

seqUAV20L = {struct('name','bike1','path','d:\data_seq\UAV123\bike1\','startFrame',1,'endFrame',3085,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
    struct('name','bird1','path','d:\data_seq\UAV123\bird1\','startFrame',1,'endFrame',2437,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
    struct('name','car1','path','d:\data_seq\UAV123\car1\','startFrame',1,'endFrame',2629,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
    struct('name','car3','path','d:\data_seq\UAV123\car3\','startFrame',1,'endFrame',1717,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
    struct('name','car6','path','d:\data_seq\UAV123\car6\','startFrame',1,'endFrame',4861,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
    struct('name','car8','path','d:\data_seq\UAV123\car8\','startFrame',1,'endFrame',2575,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
    struct('name','car9','path','d:\data_seq\UAV123\car9\','startFrame',1,'endFrame',1879,'nz',6,'ext','jpg','init_rect', [0,0,0,0]),...
  

In [25]:
import os, re, json, shutil

SRC = '/kaggle/input/datasets/galaxythereal/uav123-tracking-dataset'
DST = '/tmp/test_dataset'

# Reset UAV123 and UAV123_10fps (UAV123_20L is already good)
for name in ['UAV123', 'UAV123_10fps']:
    p = os.path.join(DST, name)
    if os.path.islink(p): os.unlink(p)
    elif os.path.isdir(p): shutil.rmtree(p)

cfg_text = open(f'{SRC}/configSeqs.m').read()

# Isolate the UAV123 block so we don't mix in seqUAV20L
m = re.search(r'seqUAV123\s*=\s*\{(.*?)\};', cfg_text, re.S)
block = m.group(1) if m else cfg_text

pattern = re.compile(
    r"struct\('name','([^']+)','path','[^']*?[\\/]([^'\\/]+)[\\/]?','startFrame',(\d+),'endFrame',(\d+),'nz',(\d+),'ext','([^']+)'"
)
seqs = pattern.findall(block)
print(f"Parsed {len(seqs)} sequences from UAV123 block")

uav123_dir = f'{DST}/UAV123'
os.makedirs(uav123_dir, exist_ok=True)
os.symlink(f'{SRC}/data_seq', f'{uav123_dir}/data_seq')
os.symlink(f'{SRC}/anno', f'{uav123_dir}/anno')

uav10_dir = f'{DST}/UAV123_10fps'
os.makedirs(f'{uav10_dir}/data_seq', exist_ok=True)
os.makedirs(f'{uav10_dir}/anno', exist_ok=True)

uav123_meta = {}
for name, folder, start_f, end_f, nz, ext in seqs:
    start_f, end_f, nz = int(start_f), int(end_f), int(nz)
    anno_path = f'{SRC}/anno/UAV123/{name}.txt'
    if not os.path.exists(anno_path):
        continue
    raw_lines = [l.strip() for l in open(anno_path) if l.strip()]
    bboxes = []
    for l in raw_lines:
        vals = [float(x) for x in l.split(',')]
        bboxes.append([int(v) if v.is_integer() else v for v in vals])

    img_names = [f'data_seq/UAV123/{folder}/{i:0{nz}d}.{ext}' for i in range(start_f, end_f + 1)]
    n = min(len(bboxes), len(img_names))
    bboxes, img_names, raw_lines = bboxes[:n], img_names[:n], raw_lines[:n]

    uav123_meta[name] = {
        'video_dir': folder,
        'init_rect': bboxes[0],
        'img_names': img_names,
        'gt_rect': bboxes,
        'attr': []
    }

    # Subsample 3x for UAV123_10fps
    seq10_dir = f'{uav10_dir}/data_seq/{name}'
    os.makedirs(seq10_dir, exist_ok=True)
    with open(f'{uav10_dir}/anno/{name}.txt', 'w') as f:
        f.write('\n'.join(raw_lines[::3]) + '\n')
    for idx, rel_img in enumerate(img_names[::3], 1):
        os.symlink(f'{SRC}/{rel_img}', f'{seq10_dir}/{idx:0{nz}d}.{ext}')

with open(f'{uav123_dir}/UAV123.json', 'w') as f:
    json.dump(uav123_meta, f)

# Verify all 3 datasets via HiFT's DatasetFactory
from toolkit.datasets import DatasetFactory
for ds_name, ds_root in [
    ('UAV123', './test_dataset/UAV123'),
    ('UAV20l', './test_dataset/UAV123_20L/data_seq'),
    ('UAV10fps', './test_dataset/UAV123_10fps/data_seq'),
]:
    ds = DatasetFactory.create_dataset(name=ds_name, dataset_root=ds_root, load_img=False)
    print(f"✅ {ds_name}: loaded {len(ds)} videos")

Parsed 123 sequences from UAV123 block


loading UAV123: 100%|██████████████████████████████████| 123/123 [00:06<00:00, 18.64it/s, person3_s]


✅ UAV123: loaded 123 videos


loading UAV20l: 100%|████████████████████████████████████████| 20/20 [00:00<00:00, 129.06it/s, uav1]


✅ UAV20l: loaded 20 videos


loading UAV10fps: 100%|██████████████████████████████| 123/123 [00:00<00:00, 164.10it/s, wakeboard9]


✅ UAV10fps: loaded 123 videos


In [28]:
%cd /kaggle/working/proj
!mkdir -p snapshot
!gdown --fuzzy "https://drive.google.com/file/d/1X8vScXvZ1QohbqzE9EAheQ_yaJCTslSA/view?usp=sharing" -O snapshot/first.pth
!ls -lh snapshot/first.pth

/kaggle/working/proj
Downloading...
From (original): https://drive.google.com/uc?id=1X8vScXvZ1QohbqzE9EAheQ_yaJCTslSA
From (redirected): https://drive.google.com/uc?id=1X8vScXvZ1QohbqzE9EAheQ_yaJCTslSA&confirm=t&uuid=4725c4c2-ca91-42f8-8ccf-6dd37cbd79c3
To: /kaggle/working/proj/snapshot/first.pth
100%|██████████████████████████████████████| 86.0M/86.0M [00:02<00:00, 30.2MB/s]
-rw-r--r-- 1 root root 83M Oct 11  2022 snapshot/first.pth


In [29]:
import pathlib

def patch(path, old, new):
    p = pathlib.Path(path)
    s = p.read_text()
    assert s.count(old) == 1, f"{path}: expected 1 match, found {s.count(old)} for:\n{old}"
    p.write_text(s.replace(old, new))

# --- 1. config: append new options (defaults = unchanged behaviour) ---
cfgp = pathlib.Path('pysot/core/config.py')
if 'TRACK.ZOOM' not in cfgp.read_text():
    with open(cfgp, 'a') as f:
        f.write('''
# Zoom experiment (test-time only). ZOOM=1.0 and INTERP='linear' reproduce the original tracker.
__C.TRACK.ZOOM = 1.0          # k>1 crops a smaller area (target looks bigger); k<1 crops a larger area
__C.TRACK.ZOOM_MODE = 'fixed' # 'fixed': k for every video; 'small': k only if first-frame sqrt(w*h) < ZOOM_THR
__C.TRACK.ZOOM_THR = 40.0     # pixels, used by ZOOM_MODE='small'
__C.TRACK.INTERP = 'linear'   # 'linear' (cv2 default) | 'cubic' | 'lanczos' | 'area' | 'auto' (cubic up, area down)
''')

# --- 2. tracker: zoom divides the crop side in init and track ---
tp = 'pysot/tracker/hift_tracker.py'
patch(tp, "    def init(self, img, bbox):", '''    def _pick_zoom(self, size):
        k = float(cfg.TRACK.ZOOM)
        if cfg.TRACK.ZOOM_MODE == 'small' and np.sqrt(size[0] * size[1]) >= cfg.TRACK.ZOOM_THR:
            return 1.0
        return k

    def init(self, img, bbox):''')

patch(tp, "        s_z = round(np.sqrt(w_z * h_z))\n",
          "        self.zoom = self._pick_zoom(self.size)   # fixed for the whole video\n"
          "        s_z = round(np.sqrt(w_z * h_z) / self.zoom)\n")

patch(tp, "        s_z = np.sqrt(w_z * h_z)\n",
          "        s_z = np.sqrt(w_z * h_z) / self.zoom\n")

# --- 3. base tracker: selectable interpolation ---
bp = 'pysot/tracker/base_tracker.py'
patch(bp, "class SiameseTracker(BaseTracker):", '''_INTERP = {
    'linear': cv2.INTER_LINEAR, 'cubic': cv2.INTER_CUBIC,
    'lanczos': cv2.INTER_LANCZOS4, 'area': cv2.INTER_AREA
}

def _pick_interp(src, dst):
    m = cfg.TRACK.INTERP
    if m == 'auto':
        return cv2.INTER_CUBIC if src < dst else cv2.INTER_AREA
    return _INTERP[m]

class SiameseTracker(BaseTracker):''')

patch(bp, "im_patch = cv2.resize(im_patch, (model_sz, model_sz))",
          "im_patch = cv2.resize(im_patch, (model_sz, model_sz),\n"
          "                                  interpolation=_pick_interp(im_patch.shape[0], model_sz))")

print("✅ Patched successfully")

✅ Patched successfully


In [31]:
import os, sys
os.chdir('/kaggle/working/proj')
if '.' not in sys.path:
    sys.path.insert(0, '.')

from pysot.core.config import cfg
cfg.merge_from_file('experiments/config.yaml')
print('INSTANCE_SIZE:', cfg.TRACK.INSTANCE_SIZE,
      '| EXEMPLAR:', cfg.TRACK.EXEMPLAR_SIZE,
      '| CONTEXT:', cfg.TRACK.CONTEXT_AMOUNT,
      '| ZOOM:', cfg.TRACK.ZOOM)

INSTANCE_SIZE: 287 | EXEMPLAR: 127 | CONTEXT: 0.5 | ZOOM: 1.0


In [33]:
!ln -sfn /tmp/test_dataset/UAV123_10fps/data_seq /tmp/test_dataset/UAV10fps
!ln -sfn /tmp/test_dataset/UAV123_20L/data_seq /tmp/test_dataset/UAV20l

In [35]:
import pathlib

p = pathlib.Path('/kaggle/working/proj/pysot/utils/model_load.py')
s = p.read_text()
old = "map_location=lambda storage, loc: storage.cuda(device))"
new = "map_location=lambda storage, loc: storage.cuda(device), weights_only=False)"

assert old in s, "Target line not found in model_load.py"
p.write_text(s.replace(old, new))
print("✅ Patched model_load.py with weights_only=False")

✅ Patched model_load.py with weights_only=False


In [ ]:
import os, yaml, copy

os.chdir('/kaggle/working/proj')
base = yaml.safe_load(open('experiments/config.yaml'))

def make_cfg(name, **trk):
    c = copy.deepcopy(base)
    c.setdefault('TRACK', {}).update(trk)
    path = f'experiments/{name}.yaml'
    yaml.safe_dump(c, open(path, 'w'))
    return path

runs = {
    'base': {},
    'z080': {'ZOOM': 0.8},
    'z090': {'ZOOM': 0.9},
    'z115': {'ZOOM': 1.15},
    'z130': {'ZOOM': 1.3},
    'z150': {'ZOOM': 1.5},
    'interp_auto': {'INTERP': 'auto'}
}

DS = 'UAV10fps'
for n, kw in runs.items():
    cfg_path = make_cfg(n, **kw)
    snap_link = f'snapshot/{n}.pth'
    if not os.path.exists(snap_link):
        os.symlink('/kaggle/working/proj/snapshot/first.pth', snap_link)
    print(f"\n=== Running {n} on {DS} ===")
    !PYTHONPATH=. python tools/test.py --dataset {DS} --snapshot {snap_link} --config {cfg_path}


=== Running base on UAV10fps ===
loading UAV10fps: 100%|██████████████████████████████| 123/123 [00:00<00:00, 178.93it/s, wakeboard9]
(  1) Video: bike1        Time:  25.3s Speed: 40.7fps
(  2) Video: bike2        Time:   3.8s Speed: 48.8fps
(  3) Video: bike3        Time:   3.0s Speed: 48.6fps
(  4) Video: bird1_1      Time:   1.8s Speed: 45.9fps
(  5) Video: bird1_2      Time:   4.8s Speed: 48.7fps
(  6) Video: bird1_3      Time:   9.0s Speed: 32.1fps
(  7) Video: boat1        Time:   6.5s Speed: 46.3fps
(  8) Video: boat2        Time:   6.0s Speed: 44.1fps
(  9) Video: boat3        Time:   6.9s Speed: 43.7fps
( 10) Video: boat4        Time:   4.0s Speed: 46.1fps
( 11) Video: boat5        Time:   3.5s Speed: 48.3fps
( 12) Video: boat6        Time:   5.6s Speed: 48.2fps
( 13) Video: boat7        Time:   3.8s Speed: 47.0fps
( 14) Video: boat8        Time:   4.7s Speed: 48.6fps
( 15) Video: boat9        Time:   9.5s Speed: 48.9fps
( 16) Video: building1    Time:   3.2s Speed: 48.6fps
(